# Preparation fluc data (figure m05)

In [ ]:
# Libraries
import os, datetime
import xarray as xr

In [ ]:
# Directories
dir01 = '../paper_ecosystem_carbon/output/01_data_prep/'
dir07 = '../paper_ecosystem_carbon/output/07_eval_prep/'
dir_out = '../paper_ecosystem_carbon/output/zenodo/'

---

In [ ]:
# Directory with processed blue data
dir07_bluep = f'{dir07}blue_processed/'

# Define blue run strings
run_chansis = 'run_20260515_default_1700-2024'
run_cspatial = 'run_20260515_cspatial_1700-2024'
run_cdgvm = 'run_20260515_cdgvm_1700-2024'

In [ ]:
# Get land mask
ds_land = xr.open_dataset(f'{dir01}ds_luh2_land_mask_r025.nc')

In [ ]:
# PFT names
pft_dict = {
    1: "Tropical evergreen forest",
    2: "Tropical deciduous forest",
    3: "Temperate evergreen broadleaf forest",
    4: "Temperate/boreal deciduous broadleaf forest",
    5: "Temperate/boreal evergreen conifers",
    6: "Temperate/boreal deciduous conifers",
    7: "Raingreen shrubs",
    8: "Summergreen shrubs",
    9: "C3 natural grasses",
    10: "C4 natural grasses",
    11: "Tundra"
}

In [ ]:
# Functions to process blue data
def get_blue_data_components(dir_in, run, 
                             chunk_time=50, chunk_lat=-1, chunk_lon=-1):
    """Returns prepared blue components data as lazy dataset."""
    ds = xr.open_mfdataset(f'{dir_in}{run}_eluc_components.nc') \
        .chunk(chunks=dict(time=chunk_time, lat=chunk_lat, lon=chunk_lon))
    
    ds['eluc_wood_harvest'] = (ds.eluc_wood_harvest_sources + 
                               ds.eluc_wood_harvest_sinks)
    ds = ds.drop_vars(
        ['eluc_wood_harvest_sources', 'eluc_wood_harvest_sinks',
         'eluc_sources', 'eluc_sinks'])

    ds = ds.rename({'eluc_forestation': 'eluc_forest_regrowth'})
    
    return ds


def get_blue_data_pft(dir_in, run, 
                      chunk_time=50, chunk_lat=-1, chunk_lon=-1):
    """Returns prepared blue pft data as lazy dataset."""  
    ds = xr.open_mfdataset(f'{dir_in}{run}_eluc_pft.nc') \
        .chunk(chunks=dict(time=chunk_time, lat=chunk_lat, lon=chunk_lon))
    
    for i in ds.pft:
        ds[f'eluc_pft{int(i)}'] = ds['eluc_pft'].sel(pft = i).drop_vars('pft') 
    
    ds = ds.drop_vars('eluc_pft').drop_dims('pft')

    return ds


def prep_ds0(dir_in, run):
    """Returns merged blue dataset with components and pft data"""
    return xr.merge(
        [get_blue_data_components(dir_in, run), 
         get_blue_data_pft(dir_in, run)]) \
        .sel(time=slice(1960, 2023)) \
        .chunk(chunks=dict(time=-1))


# Prepare fluc data
ds0_chansis = prep_ds0(dir07_bluep, run_chansis)
ds0_cspatial = prep_ds0(dir07_bluep, run_cspatial)
ds0_cdgvm = prep_ds0(dir07_bluep, run_cdgvm)

In [ ]:
# Function to update variable names and attributes
def prep_ds1(s):

    """Returns dataset with updated variable names and attributes"""
    
    if s == 'chansis':
        ds_in = ds0_chansis
        carbon_data = 'cHansis'
        run = run_chansis
    elif s == 'cspatial':
        ds_in = ds0_cspatial
        carbon_data = 'cSpatial'
        run = run_cspatial
    elif s == 'cdgvm':
        ds_in = ds0_cdgvm
        carbon_data = 'cDGVM'
        run = run_cdgvm
    else:
        raise ValueError(
            "s must be 'chansis', 'cspatial', or 'cdgvm'"
        )
    
    # Drop cell area variables
    ds = ds_in.drop_vars('cell_area')
    
    # Rename to fluc
    rename_dict = {
        var: f"{var.replace('eluc_', 'fluc_', 1)}"
        for var in ds.data_vars
    }
    
    ds = ds.rename(rename_dict)
    
    # Update attributes of variables
    long_names = {
        'fluc_net': 'net land-use change carbon flux',
        'fluc_deforestation': 'land-use change carbon flux from deforestation',
        'fluc_forest_regrowth': 'land-use change carbon flux from forest (re-)growth',
        'fluc_wood_harvest': 'land-use change carbon flux from wood harvest',
        'fluc_other_transitions': 'land-use change carbon flux from other land-use transitions',
    }
    
    for var, long_name in long_names.items():
        ds[var].attrs.update({
            'long_name': f'{long_name} ({carbon_data})',
        })
        ds[var].attrs.pop('standard_name', None)
    
    
    for i in range(1, 12):
        long_name = f'land-use change carbon flux for PFT {i} ({carbon_data})'
        ds[f'fluc_pft{i}'].attrs = {
            'long_name': long_name,
            'pft': f'{pft_dict[i]}'
        }
    
    for var in ds.data_vars:
        ds[var].attrs['units'] = 'tC ha-1 yr-1'
        ds[var].attrs['underlying_carbon_data'] = carbon_data
        ds[var].attrs['blue_simulation'] = run
    
    # Add simulation to variable names
    ds = ds.rename({var: f"{s}_{var}" for var in ds.data_vars})

    # Apply land-sea mask
    ds = ds.where(ds_land.luh2_land_mask)
    
    # Add grid cell_area
    ds['cell_area'] = ds_in['cell_area']
    ds['cell_area'].attrs = {
        'long_name': f'grid cell area',
        'units': 'ha'
        }

    # Update attributes of latitude and longitude
    ds['lat'].attrs = {
        'standard_name': 'latitude',
        'long_name': 'latitude',
        'units': 'degrees_north',
        'axis': 'Y'
    }
    
    ds['lon'].attrs = {
        'standard_name': 'longitude',
        'long_name': 'longitude',
        'units': 'degrees_east',
        'axis': 'X'
    }
    
    # Update time attributes
    ds['time'].attrs = {'long_name': 'calendar year'}
    
    return ds

ds1_chansis = prep_ds1('chansis')
ds1_cspatial = prep_ds1('cspatial')
ds1_cdgvm = prep_ds1('cdgvm')

---

In [ ]:
# Function to export dataset
def prep_export(export):

    """Returns/Exports final blue data as nc file"""
    
    # Merge prepared blue datasets
    ds_out = xr.merge([ds1_chansis.drop_vars('cell_area'), 
                       ds1_cspatial.drop_vars('cell_area'), 
                       ds1_cdgvm],
                     join='exact')
    
    # Update dataset attributes
    date = datetime.datetime.now(datetime.timezone.utc)
    ds_out.attrs.update({
        'title': 'Land-use change carbon flux estimates from BLUE',
        'institution': (
            'Department of Geography, '
            'Ludwig-Maximilians-Universität München, Munich, Germany'
        ),
        'creator_name': 'Raphael Ganzenmüller',
        'version': '1.0',
        'date_created': date.strftime('%Y-%m-%d %H:%M UTC')
    })

    # Checks
    assert 'cell_area' in ds_out
    assert len([v for v in ds_out.data_vars if '_fluc_' in v]) == 48
    
    # Create directory if missing
    os.makedirs(dir_out, exist_ok=True)
    
    # Get encoding
    encoding = {
        var: {
            'dtype': 'float32',
            '_FillValue': -999999.0,
            'zlib': True,
            'complevel': 6} 
    for var in ds_out.data_vars
    }
        
    # Export
    if export:
        file_out = f'ecosystem_carbon_data_figure_m05.nc'
        ds_out.to_netcdf(os.path.join(dir_out, file_out),
                         engine='netcdf4', encoding=encoding, mode='w')
    
    
    return ds_out


prep_export(True)

---

### Check

In [ ]:
file_path = os.path.join(dir_out, 'ecosystem_carbon_data_figure_m05.nc')

with xr.open_dataset(file_path, decode_cf=False) as ds:
    print(ds)

with xr.open_dataset(file_path, decode_cf=True) as ds:
    print(ds)

In [ ]:
# Check if general data settings are as expected
with xr.open_dataset(file_path) as ds:
    assert ds.sizes['lat'] == 720
    assert ds.sizes['lon'] == 1440
    assert ds.sizes['time'] == 64
    assert ds.time.values[0] == 1960
    assert ds.time.values[-1] == 2023
    assert len([v for v in ds.data_vars if '_fluc_' in v]) == 48

In [ ]:
# Check if yearly sums of net, components and pfts are approx. equal
with xr.open_dataset(file_path) as ds:

    def prep_ds_ysum(ds):
        """Returns yearly global values."""
        return (
            ((ds[[v for v in ds.data_vars if '_fluc_' in v]] * ds.cell_area)
            .sum(['lat', 'lon']) * 1E-9)
        ).compute()
        
    ds_ysum = prep_ds_ysum(ds)

    for s in ['chansis', 'cspatial', 'cdgvm']:
        list_comps = ['deforestation', 
                      'forest_regrowth', 
                      'other_transitions', 
                      'wood_harvest'] 
        list_pfts = [f'pft{i}' for i in range(1, 12)]
        
        ds_ysum_sel = ds_ysum[
            [var for var in ds_ysum.data_vars if var.startswith(s)]
            ]
        
        a_net = ds_ysum_sel[f'{s}_fluc_net']
        a_com = (ds_ysum_sel[[f'{s}_fluc_{i}' for i in list_comps]].to_array('a').sum('a'))
        a_pft = ds_ysum_sel[[f'{s}_fluc_{i}' for i in list_pfts]].to_array('a').sum('a')
        
        xr.testing.assert_allclose(a_com, a_net, rtol=0.01, atol=1e-6,)
        xr.testing.assert_allclose(a_pft, a_net, rtol=0.01, atol=1e-6,)
        xr.testing.assert_allclose(a_com, a_pft, rtol=0.01, atol=1e-6,)